DATA GENERATOR

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta


def generate_smart_water_minimal(
    days=1,
    interval_sec=5,
    n_sensors=5,
    attack_fraction=0.08,
    random_seed=42
):
    """
    Generates a physics-consistent smart water dataset in long format:

      timestamp: string
      sensor_id: number (1..n_sensors)
      pressure: number
      flow_rate: number
      temperature: number
      pump_power: number
      time_of_day: string
      day_of_week: string
      month: string
      pressure_mean: number
      pressure_var: number
      pump_power_optimized: number
      anomaly_detected: boolean
    """
    rng = np.random.default_rng(random_seed)
    N = int(days * 24 * 3600 / interval_sec)

    # ============================================================
    # 1. TIME AXIS
    # ============================================================
    start = datetime(2025, 1, 1)
    timestamps_dt = [start + timedelta(seconds=i * interval_sec) for i in range(N)]
    timestamps_str = [t.strftime("%Y-%m-%d %H:%M:%S") for t in timestamps_dt]

    hour_float = np.array([t.hour + t.minute / 60 + t.second / 3600 for t in timestamps_dt])
    time_of_day_str = np.array([t.strftime("%H:%M:%S") for t in timestamps_dt])  # string
    day_of_week_str = np.array([t.strftime("%A") for t in timestamps_dt])        # string (Monday, …)
    month_str = np.array([t.strftime("%B") for t in timestamps_dt])              # string (January, …)

    dow_idx = np.array([t.weekday() for t in timestamps_dt])  # 0=Mon,6=Sun
    is_weekend = (dow_idx >= 5).astype(int)

    # ============================================================
    # 2. FLOW RATE – DAILY DEMAND + WEEKEND EFFECT
    # ============================================================
    def demand_profile(h):
        morning = 0.06 * np.exp(-0.5 * ((h - 7.5) / 1.0) ** 2)
        evening = 0.07 * np.exp(-0.5 * ((h - 19.5) / 1.5) ** 2)
        night = 0.015
        return night + morning + evening

    base_flow = demand_profile(hour_float)
    base_flow[is_weekend == 1] *= 0.75  # lower demand on weekends

    # Slow random walk
    rw = np.cumsum(rng.normal(0, 0.0002, N))
    rw -= rw.min()
    rw /= (rw.max() / 0.01)
    base_flow = base_flow + rw

    flow_rate = np.zeros(N)
    flow_rate[0] = base_flow[0]
    for i in range(1, N):
        flow_rate[i] = (
            0.85 * flow_rate[i - 1]
            + 0.15 * base_flow[i]
            + rng.normal(0, 0.0005)
        )
    flow_rate = np.clip(flow_rate, 0.005, 0.14)

    # ============================================================
    # 3. TEMPERATURE – AMBIENT (USED AS SINGLE TEMPERATURE FEATURE)
    # ============================================================
    day_idx = np.arange(N) * interval_sec / 86400.0
    ambient_temperature = (
        15
        + 10 * np.sin(2 * np.pi * (day_idx / 365 - 0.25))   # yearly
        + 5 * np.sin(2 * np.pi * (hour_float / 24 - 0.25))  # daily
        + rng.normal(0, 0.3, N)
    )
    ambient_temperature = (
        pd.Series(ambient_temperature)
        .rolling(60, min_periods=1)
        .mean()
        .values
    )
    temperature = ambient_temperature.copy()

    # ============================================================
    # 4. PUMP POWER – DEMAND-RESPONSIVE WITH INERTIA
    # ============================================================
    is_peak_hour = (
        ((hour_float >= 6) & (hour_float <= 9)) |
        ((hour_float >= 18) & (hour_float <= 22))
    ).astype(int)

    demand_norm = flow_rate / flow_rate.max()
    pump_target = 30 + 280 * demand_norm + 25 * is_peak_hour

    pump_power = np.zeros(N)
    pump_power[0] = pump_target[0]
    for i in range(1, N):
        pump_power[i] = (
            0.92 * pump_power[i - 1]
            + 0.08 * pump_target[i]
            + rng.normal(0, 0.7)
        )
    pump_power = np.clip(pump_power, 20, 320)

    # ============================================================
    # 5. RESERVOIR LEVEL – INTERNAL STATE
    # ============================================================
    k_in, k_out = 0.0012, 8.0
    reservoir_level = np.zeros(N)
    reservoir_level[0] = 70.0

    for i in range(1, N):
        dR = k_in * pump_power[i] - k_out * flow_rate[i]
        reservoir_level[i] = (
            reservoir_level[i - 1]
            + dR * (interval_sec / 3600.0)
            + rng.normal(0, 0.02)
        )
        reservoir_level[i] = np.clip(reservoir_level[i], 10, 100)

    # ============================================================
    # 6. TRUE PRESSURE – PHYSICS MODEL
    #     P ≈ ALPHA * pump_power - BETA * flow_rate*100 + GAMMA * reservoir + f(T)
    # ============================================================
    ALPHA, BETA, GAMMA = 0.12, 0.8, 0.03
    temp_effect = 0.005 * (temperature - 12.0)

    true_pressure_raw = (
        ALPHA * pump_power
        - BETA * flow_rate * 100.0
        + GAMMA * reservoir_level
        + temp_effect
        + rng.normal(0, 0.03, N)
    )

    true_pressure = np.zeros(N)
    true_pressure[0] = true_pressure_raw[0]
    for i in range(1, N):
        true_pressure[i] = (
            0.7 * true_pressure[i - 1]
            + 0.3 * true_pressure_raw[i]
        )
    true_pressure = np.clip(true_pressure, 4.5, 6.5)

    # ============================================================
    # 7. MULTIPLE SENSORS – REDUNDANT PRESSURE MEASUREMENTS
    # ============================================================
    SENSOR_NOISE = 0.05
    sensors = np.zeros((N, n_sensors))
    for k in range(n_sensors):
        bias = rng.normal(0, 0.05)
        sensors[:, k] = (
            true_pressure
            + bias
            + rng.normal(0, SENSOR_NOISE, N)
        )

    # ============================================================
    # 8. FDI ATTACKS – RANDOMLY CHANGE ONE COLUMN
    #    (pressure OR pump_power OR flow_rate) WITH RANDOM VALUE
    #    anomaly_detected is per (timestamp, sensor_id)
    # ============================================================
    anomaly_per_sensor = np.zeros((N, n_sensors), dtype=bool)

    n_attack_rows = int(N * attack_fraction)
    attack_idxs = rng.choice(N, n_attack_rows, replace=False)

    for idx in attack_idxs:
        duration = min(N - idx, int(rng.integers(1, 4)))
        attacked_feature = rng.choice(["pressure", "pump_power", "flow_rate"])

        if attacked_feature == "pressure":
            sid = int(rng.integers(0, n_sensors))
            random_pressure_value = float(rng.uniform(0.0, 10.0))
            sensors[idx:idx + duration, sid] = random_pressure_value
            anomaly_per_sensor[idx:idx + duration, sid] = True

        elif attacked_feature == "pump_power":
            random_pump_value = float(rng.uniform(20.0, 320.0))
            pump_power[idx:idx + duration] = random_pump_value
            anomaly_per_sensor[idx:idx + duration, :] = True

        else:  # flow_rate
            random_flow_value = float(rng.uniform(0.005, 0.14))
            flow_rate[idx:idx + duration] = random_flow_value
            anomaly_per_sensor[idx:idx + duration, :] = True

    # ============================================================
    # 9. PRESSURE MEAN / VAR ACROSS SENSORS
    # ============================================================
    pressure_mean = sensors.mean(axis=1)
    pressure_var = sensors.var(axis=1)

    # ============================================================
    # 10. PUMP POWER OPTIMIZER – PHYSICS-INFORMED
    # ============================================================
    P_MIN, P_MAX = 4.8, 6.2
    P_TARGET = 0.5 * (P_MIN + P_MAX)

    pump_opt_raw = (P_TARGET + BETA * flow_rate * 100 - GAMMA * reservoir_level) / ALPHA
    low_res_mask = reservoir_level < 50
    high_res_mask = reservoir_level > 80

    pump_opt_raw[low_res_mask] *= 1.15
    pump_opt_raw[high_res_mask] *= 0.9
    pump_opt_raw = np.clip(pump_opt_raw, 20, 320)

    pump_power_optimized = np.zeros_like(pump_opt_raw)
    pump_power_optimized[0] = pump_opt_raw[0]
    for i in range(1, N):
        pump_power_optimized[i] = (
            0.9 * pump_power_optimized[i - 1]
            + 0.1 * pump_opt_raw[i]
        )

    # ============================================================
    # 11. LONG FORMAT WITH sensor_id
    # ============================================================
    rows = []
    for t in range(N):
        for s in range(n_sensors):
            rows.append({
                "timestamp": timestamps_str[t],
                "sensor_id": int(s + 1),
                "pressure": float(sensors[t, s]),
                "flow_rate": float(flow_rate[t]),
                "temperature": float(temperature[t]),
                "pump_power": float(pump_power[t]),
                "time_of_day": time_of_day_str[t],
                "day_of_week": day_of_week_str[t],
                "month": month_str[t],
                "pressure_mean": float(pressure_mean[t]),
                "pressure_var": float(pressure_var[t]),
                "pump_power_optimized": float(pump_power_optimized[t]),
                "anomaly_detected": bool(anomaly_per_sensor[t, s]),
            })

    df = pd.DataFrame(rows)
    return df


if __name__ == "__main__":
    df_1 = generate_smart_water_minimal(days=1, n_sensors=5)
    df_long = df_1.copy()
    out_name = "smart_water_minimal_physics_with_sensor_id.csv"
    df_1.to_csv(out_name, index=False)

    print("Saved:", out_name)
    print("Rows:", len(df_1))
    print("Columns:", len(df_1.columns))
    print(df_1.head(10))
    print(df_1.columns.tolist())

Saved: smart_water_minimal_physics_with_sensor_id.csv
Rows: 86400
Columns: 13
             timestamp  sensor_id  pressure  flow_rate  temperature  \
0  2025-01-01 00:00:00          1  6.422204   0.024746    -0.361436   
1  2025-01-01 00:00:00          2  6.517695   0.024746    -0.361436   
2  2025-01-01 00:00:00          3  6.432255   0.024746    -0.361436   
3  2025-01-01 00:00:00          4  6.496695   0.024746    -0.361436   
4  2025-01-01 00:00:00          5  6.502600   0.024746    -0.361436   
5  2025-01-01 00:00:05          1  6.441625   0.025883    -0.169722   
6  2025-01-01 00:00:05          2  6.519910   0.025883    -0.169722   
7  2025-01-01 00:00:05          3  6.534989   0.025883    -0.169722   
8  2025-01-01 00:00:05          4  6.467233   0.025883    -0.169722   
9  2025-01-01 00:00:05          5  6.527078   0.025883    -0.169722   

   pump_power time_of_day day_of_week    month  pressure_mean  pressure_var  \
0   82.000474    00:00:00   Wednesday  January       6.474290